# SHA3-256 and SHAKE-256 user payloads

These cells use **recorded hardware evidence** by default. They do not program a board.
Use the configured Python (SCU35 PMC) kernel. Fresh hardware commands are shown in Markdown and must be invoked explicitly after configuring your exact SCU35 and recovery image.
Current PL timing uses AXI Timer; native PMC timing uses `mcycle`. JTAG/UART are outside operation timing.

Compare exact digests before discussing timing. SHA3-256 has a fixed 32-byte digest; SHAKE-256 is extendable and this example requests 32 bytes. Runtime hashes are not by themselves sender authentication.

In [1]:
from pathlib import Path
import hashlib, json, sys, time
import pandas as pd
# Works from the notebook directory, lab root, or parent repository.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for base in candidates for p in (base, base / "scu35") if (p / "lab.py").is_file()), None)
assert ROOT is not None, "Launch Jupyter inside this checkout or its scu35 directory"
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "host"))
from compare import parse_capture
from lab import config
capture = parse_capture(ROOT / "evidence/timer/uart.txt", "7d1bf77bc454dd1f")
rows = pd.DataFrame(capture["rows"])
print("Recorded board capture: 31 complete rows; current Python:", sys.version.split()[0])


Recorded board capture: 31 complete rows; current Python: 3.11.9


In [2]:
from compare import payload, digest
for (algorithm, size), actual in capture["hashes"].items():
    expected = digest(algorithm, payload(size)).hex()
    assert actual.lower() == expected.lower()
    print(algorithm, size, actual)
display(rows[rows.algorithm.isin(["SHA3-256", "SHAKE-256"])])
data = payload(2048)
for name, function in [("SHA3-256", lambda: hashlib.sha3_256(data).digest()), ("SHAKE-256", lambda: hashlib.shake_256(data).digest(32))]:
    start = time.perf_counter_ns()
    for _ in range(1000): result = function()
    print(name, "fresh Python mean us:", (time.perf_counter_ns()-start)/1000/1000)

SHA3-256 16 8D9EF5E1443F0286E4FA45C5205ED2584CBC1B19ECDE4ADB2EF7E60EA6E780CA
SHA3-256 64 9323516A9ED2B789339472E38673FD74E8E802EFBB94B0B9454F0188CCB70358
SHA3-256 256 AEAE77786CF5BF35C38135490BC2A74A7C4C2931E2390609C5E7309594D15DE6
SHA3-256 1024 0E59B93677B2624D9FEDB109770B15BC0D79362F765CB56949B236B904B2601F
SHA3-256 2048 D436B7F10419537A3E575BE7F9A75BD68F5A0E2BEC716D3E0AF4268CEBB5180E
SHAKE-256 16 99AA50C37F60C16DB90EDB15DAD722CC13208E913ED11C11575B9A1823A908EF
SHAKE-256 64 3CDB24D5E4577627B277EE32921C964B4C970EF5A1090A765CB6A46904AAC155
SHAKE-256 256 7AA8F3681C612634BCE14BDC5493A4069A76A27BFAA0578D69FDA09A4026EF55
SHAKE-256 1024 589DE0DF3FF741EA044C5F69C91902A36D1F27D720DF369E9D735C215BFE2CD6
SHAKE-256 2048 34A087F01B4B733899229024B63BFD9572F3FC8BE5032954250ED1E71E8CC70B


,algorithm,case,size_bytes,repeats,copy_cycles,setup_cycles,api_cycles,check_cycles,scu35_api_mean_us,scu35_full_mean_us,scu35_operation_mean_us,scu35_min_us,scu35_max_us,status
0,SHA3-256,valid,16,20,7080,4180,43261,0,21.6305,27.2605,23.7205,21.57,21.64,0
1,SHA3-256,valid,64,20,26280,4180,43267,0,21.6335,36.8635,23.7235,21.63,21.64,0
2,SHA3-256,valid,256,20,103080,4180,43692,0,21.8460,75.4760,23.9360,21.84,21.85,0
3,SHA3-256,valid,1024,20,410280,4180,46850,0,23.4250,230.6550,25.5150,23.42,23.43,0
4,SHA3-256,valid,2048,20,819880,4180,51054,0,25.5270,437.5570,27.6170,25.51,25.54,0
5,SHAKE-256,valid,16,20,7080,4180,43312,0,21.6560,27.2860,23.7460,21.65,21.66,0
6,SHAKE-256,valid,64,20,26280,4180,43313,0,21.6565,36.8865,23.7465,21.65,21.66,0
7,SHAKE-256,valid,256,20,103080,4180,43729,0,21.8645,75.4945,23.9545,21.86,21.87,0
8,SHAKE-256,valid,1024,20,410280,4180,46885,0,23.4425,230.6725,25.5325,23.44,23.45,0
9,SHAKE-256,valid,2048,20,819880,4180,51098,0,25.5490,437.5790,27.6390,25.54,25.56,0


SHA3-256 fresh Python mean us: 6.2988
SHAKE-256 fresh Python mean us: 5.9461


## Fresh hardware: explicit step

Configure `config.json` first. Run `python lab.py hardware --pdi /path/to/build/demo.pdi` from the lab directory for the full runtime suite, or the native/boot command above for that path. Save logs, verify recovery and compare results before discussing speed. See [audit checkpoints](../docs/audit.md).